In [ ]:
# !pip install -q evaluate seqeval
# !mkdir src

In [ ]:
%%writefile src/config.py

hf_user_name = "amin-oj"
dataset_dict = {
    "path": "conll2003",
    "revision": "refs/convert/parquet"
}
model_checkpoint = "bert-base-cased"
push_to_hub=True
train_bs = 16
eval_bs = 16
eval_on_start = True
num_train_epochs = 1
gacc_steps = 2
lr=2e-5
lr_scheduler_type = "linear"
ckpt_name = f"{model_checkpoint.split("/")[-1]}-finetuned-ner-{dataset_dict['path']}-accelerate"

In [ ]:
%%writefile src/utils.py

from kaggle_secrets import UserSecretsClient
import os, torch

def hf_login():
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")


# To simplify the evaluation part, we define this function that takes predictions and labels
# and converts them to lists of strings, as our `metric` object expects.
def postprocess(predictions, labels, label_names):
    predictions = predictions.detach().cpu().clone().numpy()
    labels = labels.detach().cpu().clone().numpy()

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    return true_labels, true_predictions

def get_mp():
    bf16_supported = False
    if torch.cuda.is_available():
        try:
            bf16_supported = torch.cuda.is_bf16_supported()
        except Exception:
            bf16_supported = False

    mixed_precision = "bf16" if bf16_supported else "fp16"
    return mixed_precision

In [ ]:
%%writefile src/preprocess.py

def align_labels_with_tokens(labels, word_ids, first_token_only = False):
  new_labels = []
  current_word = None
  for word_id in word_ids:
    if word_id != current_word:
      # Start of a new word!
      current_word = word_id
      label = -100 if word_id is None else labels[word_id]
      new_labels.append(label)
    elif word_id is None:
      # Special token
      new_labels.append(-100)
    else:
      if first_token_only:
        label = -100
      else:
        # Same word as previous token
        label = labels[word_id]
        # If the label is B-XXX we change it to I-XXX
        if label % 2 == 1:
          label += 1
      new_labels.append(label)

  return new_labels

def tokenize_and_align_labels(examples, tokenizer):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        padding=False, # the default
    )
    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(i)
        new_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

In [ ]:
%%writefile src/evaluation.py

import torch, evaluate
from tqdm.auto import tqdm
from src.utils import postprocess

def run_eval(model, eval_dataloader, lab_names, accelerator):
    model.eval()
    metric = evaluate.load("seqeval")
     # We'll compute eval_loss as a true global mean using reduce()
    total_loss = torch.tensor(0.0, device=accelerator.device)
    total_count = torch.tensor(0.0, device=accelerator.device)
    pbar = tqdm(
        eval_dataloader,
        desc="Evaluation",
        disable=not accelerator.is_main_process,
        leave=False,
        position=1,
        dynamic_ncols=True,
    )
    for batch in pbar:
        with torch.no_grad():
            with accelerator.autocast():
                outputs = model(**batch)
                loss = outputs.loss

        predictions = outputs.logits.argmax(dim=-1)
        labels = batch["labels"]
        bs = labels.shape[0]
        total_loss += loss.detach() * bs
        total_count += bs

        # since two processes may have padded the inputs and labels to different shapes,
        # we need to use `accelerator.pad_across_processes()` to make the predictions and labels the same shape before calling `gather().`
        # If we don’t do this, the evaluation will either error out or hang forever.
        predictions = accelerator.pad_across_processes(predictions, dim=1, pad_index=-100)
        labels = accelerator.pad_across_processes(labels, dim=1, pad_index=-100)

        # gather_for_metrics collects tensors from ALL processes safely (handles last-batch sizes)
        preds_g = accelerator.gather_for_metrics(predictions)
        labels_g = accelerator.gather_for_metrics(labels)

        true_predictions, true_labels = postprocess(preds_g, labels_g, lab_names)
        metric.add_batch(predictions=true_predictions,references=true_labels)

    # Reduce sums across processes => global sums
    total_loss = accelerator.reduce(total_loss, reduction="sum")
    total_count = accelerator.reduce(total_count, reduction="sum")
    m = metric.compute()
    return {
        "eval_loss": (total_loss / torch.clamp(total_count, min=1.0)).item(),
        **{
            key: m[f"overall_{key}"]
            for key in ["precision", "recall", "f1", "accuracy"]
        }
    }

In [ ]:
%%writefile main.py

# Load deps
import os, torch, math
from tqdm.auto import tqdm
from huggingface_hub import HfApi, create_repo
from datasets import load_dataset, load_dataset_builder
from transformers import (
    AutoTokenizer,
    DataCollatorForTokenClassification,
    get_scheduler,
    AutoModelForTokenClassification
)
from accelerate import Accelerator
from torch.utils.data import DataLoader
from torch.optim import AdamW

from src.config import *
from src.utils import hf_login, get_mp
from src.preprocess import align_labels_with_tokens, tokenize_and_align_labels
from src.evaluation import run_eval


if __name__ == "__main__":
    # Init accelerator
    accelerator = Accelerator(
        gradient_accumulation_steps=gacc_steps,
        mixed_precision=get_mp()
    )
    
    if accelerator.is_main_process:
       accelerator.print(f"checkpoint name: {ckpt_name}")
    
    # HF Login and repo initiation
    hf_login()
    hf_api = HfApi()
    # Create the HF repo to push the model to the HF hub
    local_artifact_dir = "bert_sft_ner_local"
    HF_REPO_ID = f"{hf_user_name}/{ckpt_name}"
    create_repo(repo_id=HF_REPO_ID, exist_ok=True)
    if accelerator.is_main_process:
       accelerator.print(f"HF user: {hf_api.whoami()['name']}")
       accelerator.print(f"HF repo: {HF_REPO_ID}")

    # Load and Prep data
    with accelerator.main_process_first():
        raw_datasets = load_dataset(**dataset_dict)
        tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
        assert tokenizer.is_fast, "tokenizer must be fast."
        
        ner_feature = raw_datasets["train"].features["ner_tags"]
        label_names = ner_feature.feature.names
        #accelerator.print(f"rank: {accelerator.process_index} | labels names: {label_names}")
    
        tokenized_datasets = raw_datasets.map(
            tokenize_and_align_labels,
            batched=True,
            remove_columns=raw_datasets["train"].column_names,
            fn_kwargs={"tokenizer": tokenizer},
            desc="Running tokenizer on dataset",
        )
    
    if accelerator.is_main_process:
       accelerator.print(f"tokenized_datasets features: {tokenized_datasets["train"].features}")

    # Preparing training components
    # We’ll use the `AdamW` optimizer, which is like `Adam`, but with a fix in the way weight decay is applied.
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
    
    train_dataloader = DataLoader(
        tokenized_datasets["train"],
        shuffle=True,
        collate_fn=data_collator,
        batch_size=train_bs,
    )
    eval_dataloader = DataLoader(
        tokenized_datasets["validation"], collate_fn=data_collator, batch_size=eval_bs
    )
    
    id2label = {i: label for i, label in enumerate(label_names)}
    label2id = {v: k for k, v in id2label.items()}
    model = AutoModelForTokenClassification.from_pretrained(
        model_checkpoint,
        id2label=id2label,
        label2id=label2id,
    )
    
    optimizer = AdamW(model.parameters(), lr=lr)

    # Wrap objects with accelerate prepare
    model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader, eval_dataloader
    )
    if accelerator.is_main_process:
        accelerator.print(f"# GPUs: {accelerator.num_processes}")
        effective_bs = train_bs * gacc_steps * accelerator.num_processes
        accelerator.print(f"effective_batch_size: {effective_bs}")
        # TODO: how to run this on a TPU??


    # Remember that we should always measure the length of the dataloader after preparing it with accelerate
    #, as that method will change its length.
    # After preparation, train_dataloader length is per-process, which is what we want for step counting.
    num_update_steps_per_epoch = math.ceil(len(train_dataloader) / gacc_steps)
    num_training_steps = num_train_epochs * num_update_steps_per_epoch
    logging_steps = eval_steps = max(1, num_update_steps_per_epoch // 4)

    lr_scheduler = get_scheduler(
        lr_scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )

    accelerator.wait_for_everyone()
    rank = accelerator.process_index
    is_main = accelerator.is_main_process
    proc_id_str = f"[rank={rank} main={is_main}]"
    accelerator.print(
        f"{proc_id_str} train_len={len(train_dataloader)} eval_len={len(eval_dataloader)}"
    )
    accelerator.print(
        f"{proc_id_str} eval_steps={eval_steps} total_opt_steps={num_training_steps}"
    )
    accelerator.wait_for_everyone()

    # Training loop
    global_step = 0
    running_loss = 0.0
    running_mb = 0
    scheduler_steps = 0
    best_eval_loss = float("inf")
    best_dir = None  # path to best checkpoint
    progress_bar = tqdm(
        total=num_training_steps,
        desc="Optimizer Updates",
        disable=not accelerator.is_main_process,
        leave=True,
        position=0,
        dynamic_ncols=True,
    )

    if eval_on_start:
        eval_logs = run_eval(model, eval_dataloader, label_names, accelerator)
        if accelerator.is_main_process:
           accelerator.print(f"Step {global_step} eval:", eval_logs)
    
    for epoch in range(num_train_epochs):
        model.train()
        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(model):
                with accelerator.autocast():
                    outputs = model(**batch)
                    loss = outputs.loss
                running_loss += loss.detach().float().item()
                running_mb += 1
                accelerator.backward(loss)
                if accelerator.sync_gradients: # only on accumulation boundary
                    # `accumulate` handles this automatically.
                    # In fact, the collective `step` call only happens on the accumulation boundary
                    # However, this is cleaner, and other steps, such as grad clipping, can be added.
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    lr_scheduler.step()
                    scheduler_steps += 1
            
            # We count "global_step" in terms of optimizer updates
            if accelerator.sync_gradients:
                global_step += 1
                progress_bar.update(1)

                # TRAIN logging (main process only)
                if global_step % logging_steps == 0:
                    avg_train_loss = running_loss / max(1, running_mb)
                    lr_val = optimizer.param_groups[0]["lr"]
                
                    # gather across ranks -> tensors of shape [num_processes]
                    loss_t = torch.tensor([avg_train_loss], device=accelerator.device)
                    lr_t   = torch.tensor([lr_val], device=accelerator.device)
                    loss_all = accelerator.gather(loss_t).detach().cpu().tolist()
                    lr_all   = accelerator.gather(lr_t).detach().cpu().tolist()
                    # Only main process logs
                    if accelerator.is_main_process:
                        for r, (l, lr_) in enumerate(zip(loss_all, lr_all)):
                            accelerator.print(
                                f"train_loss/rank_{r} = {l} | learning_rate/rank_{r} = {lr_}"
                            )
                    running_loss = 0.0
                    running_mb = 0

                # EVAL logging
                if (global_step % eval_steps == 0) or (global_step == num_training_steps):
                    eval_logs = run_eval(model, eval_dataloader, label_names, accelerator)
                    if accelerator.is_main_process:
                       accelerator.print(f"Step {global_step} eval:", eval_logs)
            # else:accelerator.print("skipped global step update") # just to make sure this works :)))
    if accelerator.is_main_process:
       accelerator.print("Scheduler steps:", scheduler_steps, "Expected:", num_training_steps)
    accelerator.wait_for_everyone()
    progress_bar.close()
    accelerator.end_training() # Close trackers
    accelerator.free_memory() # Free up memory

In [ ]:
# ! python main.py
! accelerate launch --num_processes 2 main.py

In [ ]:
# TODOs:
# - create a trainer class if it leads to a better design
# - add 8bitadam option + grad clip option
# - add wandb
# - add checkpoint tracking and save the best checkpoint
# - Apply the best version of the trainer script to all NLP tasks SFT notebooks
# - The eval progress bar is reprinted for each step. fix that

In [ ]:
        # # Save and upload
        # accelerator.wait_for_everyone()
        # unwrapped_model = accelerator.unwrap_model(model)
        # unwrapped_model.save_pretrained(local_artifact_dir, save_function=accelerator.save)
        # if accelerator.is_main_process:
        #     tokenizer.save_pretrained(local_artifact_dir)
        #     hf_api.upload_folder(
        #         repo_id=HF_REPO_ID,
        #         folder_path=local_artifact_dir,
        #         repo_type="model",
        #         commit_message=f"Training in progress epoch {epoch}",
        #         run_as_future=True # tell the HF Hub library to push in an asynchronous process in the background
        #     )